## Text Extraction Test


In [12]:
from implementation import extract_text
from fastwarc.warc import ArchiveIterator, WarcRecordType

warc_file_path = "/home/kantas/koa_scratch/ece405-assignment2-data/data/CC-MAIN-20250417135010-20250417165010-00065.warc.gz"

# Run the extract_text function on one exact WARC file and print the output
with open(warc_file_path, "rb") as f:
    for i, record in enumerate(ArchiveIterator(f, record_types=WarcRecordType.response)):
        html_bytes = record.reader.read()
        url = record.headers.get("WARC-Target-URI", "Unknown URL")
        text = extract_text(html_bytes)
        print(f"URL: {url}\nExtracted Text:\n{text}\n{'-'*80}\n")
        break

URL: http://0371rykj.com/ipfhsb/34.html
Extracted Text:
久久久久女人精品毛片,99久久精品无码一区二区毛片,被老外的又粗又大日出了水,一边吃奶一边哭乱抻又乱扭

        • <th id="gckmo"></th>
        • <ul id="gckmo"><center id="gckmo"></center></ul>
      •  
  
 
 
 
         
   
         
    
         
    
        恒溫恒濕試驗(yàn)箱
        
	在線(xiàn)咨詢(xún)
    
         
    
         
      淋雨試驗(yàn)箱 
     
        上海林頻儀器股份有限公司Shanghai Linpin Instrument Stock Co Ltd
         
     
        服務(wù)熱線(xiàn)：4000 662 888 
         手機(jī)咨詢(xún)：13818467052
        
    
         
    
         
     
           
	
      
        • 首頁(yè)
          •  
	
	
        • 
	  林頻產(chǎn)品

	  
            
		
          • 試驗(yàn)箱系列
          • 老化箱系列
          • 非標(biāo)定制系列
          • ip防護(hù)系列
          • 振動(dòng)跌落系列
            •  
	  
          

	  
        • 
	  成功案例

	  
            
		 
	  
          

	  
        • 
	  新聞中心

	  
            
		
          • 林頻新聞
          • 行業(yè)新聞
          • 常見(jiàn)問(wèn)題
            •  
	  
     

## FastText Download

In [1]:
import os
import urllib.request
import fasttext

In [2]:
model_url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
model_path = "var/lid.176.bin"
os.makedirs(os.path.dirname(model_path), exist_ok=True)

if not os.path.exists(model_path):
    print(f"Downloading FastText model from {model_url}...")
    urllib.request.urlretrieve(model_url, model_path)
    print("Download complete")
else:
    print(f"Model already exists at {model_path}, skipping download.")


Download complete


## Language Identification Test

In [3]:
from implementation import extract_text, language_identification
from fastwarc.warc import ArchiveIterator, WarcRecordType
import random

warc_file_path = "/home/kantas/koa_scratch/ece405-assignment2-data/data/CC-MAIN-20250417135010-20250417165010-00065.warc.gz"

# Count total records in the WARC file
total = 0
with open(warc_file_path, "rb") as f:
    for _ in ArchiveIterator(f, record_types=WarcRecordType.response):
        total += 1

# Pick Random 20 indicies
indices = set(random.sample(range(total), min(20, total)))

# Process only selected records
with open(warc_file_path, "rb") as f:
    for i, record in enumerate(ArchiveIterator(f, record_types=WarcRecordType.response)):
        if i in indices:
            html_bytes = record.reader.read()
            url = record.headers.get("WARC-Target-URI", "Unknown URL")
            text = extract_text(html_bytes)
            
            lines = [line for line in text.splitlines() if line.strip()][:5]
            preview = "\n".join(lines)

            language, confidence = language_identification(text)
            print(f"URL: {url}")
            print(f"Language: {language} ({confidence:.4f})")
            print(f"Preview:\n{preview}")
            print(f"{'-'*80}\n")

URL: https://almajlesnews.com/archives/7378
Language: ar (0.9980)
Preview:
Skip to content
  • Home
  • أخبار رياضية
  • أخبار دولية
  • أخبار لبنان
--------------------------------------------------------------------------------

URL: https://backtothebasket.com/products/amine-x-nb-740-benson-tech-3
Language: en (0.3918)
Preview:
 Skip to content
Free USA Shipping on Orders > $100
  • Home
  • Shop
    • Sneakers
--------------------------------------------------------------------------------

URL: https://colompo.com/courses/%D8%AA%D8%AF%D8%B1%D9%8A%D8%A8%D8%A7%D8%AA-%D9%88%D8%AA%D9%85%D8%A7%D8%B1%D9%8A%D9%86-%D9%85%D8%AC%D8%A7%D9%86%D9%8A%D8%A9-%D8%B9%D9%84%D9%89-%D8%AC%D8%AF%D9%88%D9%84-%D8%A7%D9%84%D8%B6%D8%B1%D8%A8/
Language: ar (0.5729)
Preview:
Colompo
  • Home
  • Courses
  • About us
  • Shop
--------------------------------------------------------------------------------

URL: https://flex.kvhs-bitburg-pruem.de/Veranstaltung/cmx665eeb4cd215b.html
Language: de (0.9648)
Previe

## Mask PII Test

In [1]:
from implementation import extract_text, mask_emails, mask_phone_numbers, mask_IP_addresses
from fastwarc.warc import ArchiveIterator, WarcRecordType
import random
import os

warc_file_path = "/home/kantas/koa_scratch/ece405-assignment2-data/data/CC-MAIN-20250417135010-20250417165010-00065.warc.gz"
output_review_file = "pii_review_examples.txt"

# Store tuples of (url, original_text, masked_text, counts)
documents_with_pii = []

print("Scanning WARC file for PII replacements...")
with open(warc_file_path, "rb") as f:
    for i, record in enumerate(ArchiveIterator(f, record_types=WarcRecordType.response)):
        # Stop early if we have found plenty of examples to sample from
        if len(documents_with_pii) > 100: 
            break
            
        try:
            html_bytes = record.reader.read()
            url = record.headers.get("WARC-Target-URI", "Unknown URL")
            text = extract_text(html_bytes)
            if not text.strip():
                continue

            email_masked, email_count = mask_emails(text)
            phone_masked, phone_count = mask_phone_numbers(email_masked)
            ip_masked, ip_count = mask_IP_addresses(phone_masked)

            total_replacements = email_count + phone_count + ip_count

            # Only save records where at least one replacement was made
            if total_replacements > 0:
                counts = f"Emails: {email_count}, Phones: {phone_count}, IPs: {ip_count}"
                documents_with_pii.append((url, text, ip_masked, counts))
                print(f"Found PII in record {i}...")
                
        except Exception as e:
            continue

print(f"\nFinished scanning. Found {len(documents_with_pii)} documents with PII.")

# Pick exactly 20 random examples from the ones that actually have PII
sample_size = min(20, len(documents_with_pii))
random_20 = random.sample(documents_with_pii, sample_size)

with open(output_review_file, "w", encoding="utf-8") as out_f:
    for idx, (url, orig_text, masked_text, counts) in enumerate(random_20):
        out_f.write(f"=== Example {idx + 1} ===\n")
        out_f.write(f"URL: {url}\n")
        out_f.write(f"Replacements: {counts}\n\n")
        out_f.write("--- MASKED TEXT ---\n")
        out_f.write(masked_text + "\n")
        out_f.write("="*80 + "\n\n")

print(f"Saved {sample_size} random examples to {output_review_file}. Open this file to look for false positives/negatives!")

Scanning WARC file for PII replacements...
Found PII in record 0...
Found PII in record 1...
Found PII in record 3...
Found PII in record 4...
Found PII in record 7...
Found PII in record 11...
Found PII in record 15...
Found PII in record 16...
Found PII in record 17...
Found PII in record 18...
Found PII in record 19...
Found PII in record 21...
Found PII in record 24...
Found PII in record 26...
Found PII in record 27...
Found PII in record 29...
Found PII in record 30...
Found PII in record 31...
Found PII in record 32...
Found PII in record 33...
Found PII in record 34...
Found PII in record 35...
Found PII in record 38...
Found PII in record 39...
Found PII in record 40...
Found PII in record 41...
Found PII in record 42...
Found PII in record 44...
Found PII in record 45...
Found PII in record 46...
Found PII in record 48...
Found PII in record 49...
Found PII in record 51...
Found PII in record 52...
Found PII in record 53...
Found PII in record 56...
Found PII in record 58...


## Harmful content

In [2]:
import os
import random
from fastwarc.warc import ArchiveIterator, WarcRecordType
from implementation import extract_text, classify_NSFW, classify_toxic_speech

def evaluate_harmful_content(warc_path: str, output_file: str, sample_size: int = 20):
    print(f"Scanning {warc_path} for harmful content evaluation...")
    
    evaluated_documents = []
    
    with open(warc_path, "rb") as stream:
        for i, record in enumerate(ArchiveIterator(stream, record_types=WarcRecordType.response)):
            if len(evaluated_documents) >= 100:
                break
                
            try:
                html_bytes = record.reader.read()
                url = record.headers.get("WARC-Target-URI", "Unknown URL")
                text = extract_text(html_bytes)
                
                # Skip empty documents
                if not text.strip():
                    continue
                
                # Run the classifiers
                nsfw_label, nsfw_conf = classify_NSFW(text)
                toxic_label, toxic_conf = classify_toxic_speech(text)
                
                # Store the results
                evaluated_documents.append({
                    "url": url,
                    "text": text,
                    "nsfw_label": nsfw_label,
                    "nsfw_conf": nsfw_conf,
                    "toxic_label": toxic_label,
                    "toxic_conf": toxic_conf
                })
                
                if len(evaluated_documents) % 25 == 0:
                    print(f"Evaluated {len(evaluated_documents)} documents...")
                    
            except Exception as e:
                # Silently skip records that fail to parse
                continue

    print(f"\nFinished scanning. Selecting {sample_size} random examples for manual review.")
    
    # Randomly select the required number of examples
    random_sample = random.sample(evaluated_documents, min(sample_size, len(evaluated_documents)))
    
    # Write the output to a text file for easy manual review
    with open(output_file, "w", encoding="utf-8") as out_f:
        out_f.write("=== HARMFUL CONTENT MANUAL REVIEW ===\n\n")
        
        for idx, doc in enumerate(random_sample):
            out_f.write(f"--- Example {idx + 1} ---\n")
            out_f.write(f"URL: {doc['url']}\n")
            out_f.write(f"NSFW Prediction: {doc['nsfw_label']} (Confidence: {doc['nsfw_conf']:.4f})\n")
            out_f.write(f"Toxic Prediction: {doc['toxic_label']} (Confidence: {doc['toxic_conf']:.4f})\n\n")
            
            # We truncate to avoid massive text blocks making the file unreadable
            text_snippet = doc['text'][:1000].replace('\n', ' ') + ("..." if len(doc['text']) > 1000 else "")
            out_f.write(f"Text Snippet: {text_snippet}\n\n")
            out_f.write("="*80 + "\n\n")
            
    print(f"Saved review file to {output_file}. Please open this file to formulate your response.")

if __name__ == "__main__":
    warc_file = "/home/kantas/koa_scratch/ece405-assignment2-data/data/CC-MAIN-20250417135010-20250417165010-00065.warc.gz"
    output_txt = "harmful_content_review.txt"
    
    evaluate_harmful_content(warc_file, output_txt)

Scanning /home/kantas/koa_scratch/ece405-assignment2-data/data/CC-MAIN-20250417135010-20250417165010-00065.warc.gz for harmful content evaluation...
Evaluated 25 documents...
Evaluated 50 documents...
Evaluated 75 documents...
Evaluated 100 documents...

Finished scanning. Selecting 20 random examples for manual review.
Saved review file to harmful_content_review.txt. Please open this file to formulate your response.


## Gopher Test

In [3]:
import os
import random
from fastwarc.warc import ArchiveIterator, WarcRecordType
from implementation import extract_text, gopher_quality_filter

def evaluate_gopher_quality(warc_path: str, output_file: str, sample_size: int = 20):
    print(f"Scanning {warc_path} for Gopher quality evaluation...")
    
    evaluated_documents = []
    
    with open(warc_path, "rb") as stream:
        for i, record in enumerate(ArchiveIterator(stream, record_types=WarcRecordType.response)):
            # Grab a pool of 100 documents to randomly sample from
            if len(evaluated_documents) >= 100:
                break
                
            try:
                html_bytes = record.reader.read()
                url = record.headers.get("WARC-Target-URI", "Unknown URL")
                text = extract_text(html_bytes)
                
                # Skip completely empty documents
                if not text.strip():
                    continue
                
                # Run the Gopher quality filter
                passed_gopher = gopher_quality_filter(text)
                
                # Store the results
                evaluated_documents.append({
                    "url": url,
                    "text": text,
                    "passed_gopher": passed_gopher
                })
                
                if len(evaluated_documents) % 25 == 0:
                    print(f"Evaluated {len(evaluated_documents)} documents...")
                    
            except Exception as e:
                # Silently skip records that fail to parse
                continue

    print(f"\nFinished scanning. Selecting {sample_size} random examples for manual review.")
    
    # Randomly select exactly 20 examples
    random_sample = random.sample(evaluated_documents, min(sample_size, len(evaluated_documents)))
    
    # Write the output to a text file
    with open(output_file, "w", encoding="utf-8") as out_f:
        out_f.write("=== GOPHER QUALITY FILTER MANUAL REVIEW ===\n\n")
        
        for idx, doc in enumerate(random_sample):
            out_f.write(f"--- Example {idx + 1} ---\n")
            out_f.write(f"URL: {doc['url']}\n")
            
            # Format the prediction clearly
            status = "PASSED (High Quality)" if doc['passed_gopher'] else "FAILED (Low Quality)"
            out_f.write(f"Gopher Prediction: {status}\n\n")
            
            # Print a larger snippet (up to 1500 chars) since quality is hard to judge from just a few words
            text_snippet = doc['text'][:1500].replace('\n', ' ') + ("..." if len(doc['text']) > 1500 else "")
            out_f.write(f"Text Snippet:\n{text_snippet}\n\n")
            out_f.write("="*80 + "\n\n")
            
    print(f"Saved review file to {output_file}. Please open this to write your deliverable response.")

if __name__ == "__main__":
    warc_file = "/home/kantas/koa_scratch/ece405-assignment2-data/data/CC-MAIN-20250417135010-20250417165010-00065.warc.gz"
    output_txt = "gopher_review.txt"
    
    evaluate_gopher_quality(warc_file, output_txt)

Scanning /home/kantas/koa_scratch/ece405-assignment2-data/data/CC-MAIN-20250417135010-20250417165010-00065.warc.gz for Gopher quality evaluation...
Evaluated 25 documents...
Evaluated 50 documents...
Evaluated 75 documents...
Evaluated 100 documents...

Finished scanning. Selecting 20 random examples for manual review.
Saved review file to gopher_review.txt. Please open this to write your deliverable response.


## Train Text


In [ ]:
import fasttext

def train_quality_model():
    print("Starting FastText training on train_shuffled.txt...")
    
    # Train the model
    model = fasttext.train_supervised(
        input="/home/kantas/koa_scratch/ece405-assignment2-data/data/train_shuffled.txt", 
        epoch=25, 
        wordNgrams=2,
        lr=0.1
    )
    
    # Save the trained model to your var folder
    model_path = "/home/kantas/koa_scratch/ece405-assignment2-data/cs336-data/cs336_data/var/quality_classifier.bin"
    model.save_model(model_path)
    
    print(f"Success. Model saved to {model_path}")

    # Test the model on the training data to see the metrics
    samples, precision, recall = model.test("/home/kantas/koa_scratch/ece405-assignment2-data/data/train_shuffled.txt")
    print(f"Training Stats - Samples: {samples}, Precision: {precision:.3f}, Recall: {recall:.3f}")

if __name__ == "__main__":
    train_quality_model()

Starting FastText training on train_shuffled.txt...


Read 4M words
Number of words:  332858
Number of labels: 2
Progress:  99.5% words/sec/thread:  643860 lr:  0.000509 avg.loss:  0.410488 ETA:   0h 0m 0s

Success. Model saved to /home/kantas/koa_scratch/ece405-assignment2-data/cs336-data/cs336_data/var/quality_classifier.bin
Training Stats - Samples: 3076, Precision: 0.910, Recall: 0.910


Progress: 100.0% words/sec/thread:  641113 lr:  0.000000 avg.loss:  0.410930 ETA:   0h 0m 0s
